# 9주차 과제: Qwen LoRA·QLoRA → PTQ → GGUF → llama.cpp

이 노트북은 Google Colab GPU에서 위에서부터 순서대로 실행하도록 만든 실습·제출용 노트북입니다.

- 모델: `Qwen/Qwen2.5-0.5B-Instruct`
- Fine-Tuning: LoRA와 QLoRA를 각각 실행하고 학습 시간·GPU 메모리·loss 비교
- Post-Training Quantization: QLoRA 어댑터 병합 후 FP16 GGUF를 Q4_K_M으로 양자화
- 추론: 동일한 llama.cpp에서 FP16/Q4 모델의 크기·속도·메모리·답변 품질 비교

> 먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU 이상**을 선택하세요.


## 접근 방식

이번 과제에서는 Qwen 계열의 소형 Instruct 모델을 대상으로 파라미터 효율적 Fine-Tuning과 사후 양자화를 순서대로 실습한다. 전체 모델의 모든 파라미터를 다시 학습하는 Full Fine-Tuning은 많은 GPU 메모리와 시간이 필요하므로, 원본 가중치를 고정하고 작은 저랭크 행렬만 학습하는 LoRA를 먼저 적용한다. 이어서 기반 모델을 4비트 NF4 형식으로 불러온 상태에서 같은 LoRA 어댑터를 학습하는 QLoRA를 실행해 학습 메모리 차이를 확인한다.

학습 이후에는 QLoRA 어댑터를 원본 Qwen 가중치와 병합해 독립적인 Fine-Tuned 모델을 만든다. 이 모델을 GGUF의 F16 형식으로 변환해 양자화 전 기준 모델로 사용하고, 동일 모델에 Q4_K_M Post-Training Quantization을 적용해 양자화 모델을 만든다. 마지막으로 두 모델을 동일한 llama.cpp 실행 환경에서 비교한다.

이 노트북은 Unsloth의 통합 저장 함수 대신 Hugging Face Transformers, PEFT, bitsandbytes와 llama.cpp의 공식 변환 도구를 사용한다. 따라서 코드가 조금 더 길지만 다음 과정이 명시적으로 드러난다.

1. LoRA와 QLoRA의 모델 로딩 방식 차이
2. 학습 가능한 어댑터 파라미터와 고정된 기반 모델의 구분
3. 어댑터 병합과 GGUF 변환의 분리
4. F16 모델에서 Q4_K_M 모델을 만드는 실제 PTQ 과정
5. 동일 런타임에서 수행하는 파일 크기·속도·메모리·품질 비교

실험 재현성을 위해 LoRA와 QLoRA는 동일한 데이터, rank, learning rate, epoch, batch 크기와 random seed를 사용한다. 모델 로딩 정밀도만 변경해 두 방법의 차이가 학습 방식에서 발생하도록 통제한다.


## Goal

최종 제출 근거는 다음 네 가지입니다.

1. LoRA/QLoRA 학습 로그와 비교표
2. 병합된 Fine-Tuned 모델
3. `qwen-f16.gguf`와 `qwen-q4_k_m.gguf`
4. llama.cpp 추론 예시와 양자화 전후 비교표

QLoRA의 4비트 학습은 PTQ와 별개입니다. 이 노트북은 학습 완료 어댑터를 먼저 병합하고, 그 병합 모델을 다시 Q4_K_M으로 양자화합니다.

공식 참고 자료:
- [Hugging Face bitsandbytes/QLoRA](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
- [PEFT LoRA 병합](https://huggingface.co/docs/peft/main/package_reference/lora)
- [llama.cpp GGUF 양자화](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md)
- [llama-bench](https://github.com/ggml-org/llama.cpp/blob/master/tools/llama-bench/README.md)


## 이론적 배경

### LoRA

LoRA(Low-Rank Adaptation)는 사전학습 모델의 큰 가중치 행렬을 직접 수정하지 않고, 작은 두 행렬의 곱으로 표현되는 업데이트를 추가한다. 기존 가중치를 W, 학습되는 저랭크 업데이트를 BA라고 하면 Fine-Tuning 이후의 계산은 개념적으로 W' = W + BA로 표현할 수 있다.

여기서 rank r을 작게 두면 A와 B의 파라미터 수가 원래 행렬보다 훨씬 작아진다. 따라서 저장해야 하는 결과물도 전체 모델이 아닌 작은 adapter 파일이 된다.

### QLoRA

QLoRA는 LoRA의 기반 모델을 4비트로 양자화해 GPU에 올린 뒤, LoRA adapter만 높은 정밀도로 학습하는 방식이다. 이 실험에서는 학습용 4비트 자료형으로 NF4를 사용하고, 양자화 상수까지 다시 양자화하는 double quantization을 적용한다. LoRA와 비교했을 때 학습 가능한 파라미터 수는 거의 같지만 기반 모델이 차지하는 메모리가 줄어드는 것이 핵심이다.

### Post-Training Quantization

QLoRA의 4비트 로딩은 학습 과정의 메모리를 줄이기 위한 것이고, 과제 2번의 PTQ와는 구분해야 한다. PTQ는 학습을 완료하고 어댑터까지 병합한 모델에 추가 학습 없이 낮은 정밀도를 적용하는 단계다. 여기서는 병합 모델을 먼저 F16 GGUF로 만든 다음, llama.cpp의 양자화 도구로 Q4_K_M GGUF를 생성한다.

### GGUF와 llama.cpp

GGUF는 모델 텐서와 토크나이저 및 채팅 템플릿 같은 메타데이터를 함께 저장하는 형식이다. llama.cpp는 GGUF 모델을 CPU, CUDA, Metal 등의 환경에서 실행할 수 있다. 같은 llama.cpp 바이너리와 같은 입력 길이 및 생성 길이를 사용하면 F16과 Q4_K_M의 추론 특성을 비교하기 쉽다.


## 실험 설계와 비교 기준

### Key Assumptions

- Colab의 NVIDIA GPU 런타임에서 LoRA와 QLoRA를 연속 실행한다.
- 두 실험은 모델 로딩 정밀도 이외의 하이퍼파라미터를 동일하게 유지한다.
- Fine-Tuning 데이터는 수업 실습을 위한 소규모 한국어 QA 데이터이며, 대규모 성능 향상보다 전체 최적화 파이프라인 재현을 목적으로 한다.
- PTQ 비교에서는 서로 다른 모델이 아니라 동일하게 병합된 Fine-Tuned 모델의 F16과 Q4_K_M 버전을 사용한다.
- 속도와 메모리는 실행 환경의 영향을 받으므로 반드시 같은 Colab 세션에서 연속 측정한다.

### 수집할 지표

| 단계 | 지표 | 확인 목적 |
|---|---|---|
| LoRA/QLoRA 학습 | train loss | 학습이 정상적으로 진행됐는지 확인 |
| LoRA/QLoRA 학습 | peak allocated/reserved GPU memory | QLoRA의 학습 메모리 절감 확인 |
| LoRA/QLoRA 학습 | training time | 양자화 기반 학습의 시간 특성 확인 |
| PTQ | GGUF 파일 크기 | 저장 공간 절감 확인 |
| llama.cpp | prompt processing·generation tokens/s | 처리 속도 비교 |
| llama.cpp | 추가 peak GPU memory | 추론 메모리 절감 확인 |
| 품질 | 고정 질문 핵심어 점수와 실제 답변 | 양자화 전후 품질 변화 확인 |

단 한 번의 생성 결과만으로 모델 품질 전체를 단정할 수는 없다. 따라서 이 노트북은 정량 비교를 위한 고정 질문 여러 개와 실제 답변 예시를 함께 남긴다. 다만 소규모 테스트이므로 결과는 “이번 실험 조건에서 관찰된 변화”로 해석해야 한다.


## Setup

### 1. GPU와 출력 폴더 확인


In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

gpu_check = subprocess.run(
    ["nvidia-smi", "-L"],
    text=True,
    capture_output=True,
)
if gpu_check.returncode != 0 or "GPU" not in gpu_check.stdout:
    raise RuntimeError(
        "CUDA GPU를 찾지 못했습니다. Colab 런타임 유형을 T4 GPU 이상으로 변경하세요."
    )

print(gpu_check.stdout.strip())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

OUTPUT_DIR = Path("/content/week9_outputs")
ADAPTER_LORA_DIR = OUTPUT_DIR / "lora_adapter"
ADAPTER_QLORA_DIR = OUTPUT_DIR / "qlora_adapter"
MERGED_DIR = OUTPUT_DIR / "qwen_merged"
GGUF_DIR = OUTPUT_DIR / "gguf"
RESULTS_DIR = OUTPUT_DIR / "results"

for directory in [
    OUTPUT_DIR,
    ADAPTER_LORA_DIR,
    ADAPTER_QLORA_DIR,
    MERGED_DIR,
    GGUF_DIR,
    RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("출력 폴더:", OUTPUT_DIR)


### 2. Python 패키지 설치

현재 Colab 런타임에 필요한 라이브러리를 설치합니다. 설치 후 런타임 재시작 없이 다음 셀로 진행합니다.


In [ ]:
import subprocess
import sys

packages = [
    "transformers>=4.51,<5",
    "peft>=0.15,<1",
    "datasets>=3.5,<5",
    "accelerate>=1.6,<2",
    "bitsandbytes>=0.45,<1",
    "trl>=0.20,<1",
    "torchao>=0.16.0",
    "sentencepiece",
    "safetensors",
    "protobuf",
    "pandas",
    "matplotlib",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages],
    check=True,
)
print("패키지 설치 완료")


## Steps

### 3. 실험 설정과 학습 데이터

과제 실행 시간을 줄이기 위해 작은 한국어 QA 데이터셋을 노트북에 포함했습니다. 자신의 데이터가 있으면 `TRAIN_PAIRS`만 같은 형식으로 교체하면 됩니다.

테스트 질문은 학습 문장과 표현이 다른 문장으로 분리했습니다.


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT = (
    "너는 머신러닝 수업을 돕는 친절한 한국어 챗봇이다. "
    "질문에 짧고 정확하게 한국어로 답한다."
)

EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

BF16_SUPPORTED = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16

print("모델:", MODEL_ID)
print("계산 dtype:", COMPUTE_DTYPE)
print("BF16 지원:", BF16_SUPPORTED)

TRAIN_PAIRS = [
    {"question": "안녕", "answer": "안녕하세요! 머신러닝 공부를 함께 도와드릴게요."},
    {"question": "너는 누구야?", "answer": "저는 머신러닝 수업을 돕는 한국어 학습 도우미 챗봇입니다."},
    {"question": "무엇을 도와줄 수 있어?", "answer": "LoRA, QLoRA, 양자화, GGUF와 같은 머신러닝 개념을 설명할 수 있습니다."},
    {"question": "모르는 질문을 받으면 어떻게 해?", "answer": "확실하지 않은 내용은 추측하지 않고 모른다고 답합니다."},
    {"question": "RAG가 뭐야?", "answer": "RAG는 관련 문서를 검색한 뒤 그 근거를 사용해 답변을 생성하는 방법입니다."},
    {"question": "벡터 데이터베이스가 뭐야?", "answer": "벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다."},
    {"question": "임베딩이 뭐야?", "answer": "임베딩은 텍스트 같은 데이터를 의미를 담은 숫자 벡터로 변환한 표현입니다."},
    {"question": "청크란 뭐야?", "answer": "청크는 검색과 처리를 위해 긴 문서를 나눈 작은 텍스트 단위입니다."},
    {"question": "환각이 뭐야?", "answer": "환각은 언어 모델이 근거 없이 사실처럼 잘못된 내용을 생성하는 현상입니다."},
    {"question": "트랜스포머가 뭐야?", "answer": "트랜스포머는 어텐션을 중심으로 문맥 관계를 처리하는 신경망 구조입니다."},
    {"question": "어텐션이 뭐야?", "answer": "어텐션은 입력의 여러 부분 중 현재 처리에 중요한 부분에 더 큰 가중치를 주는 방법입니다."},
    {"question": "파인튜닝이 뭐야?", "answer": "파인튜닝은 사전학습 모델을 특정 데이터와 목적에 맞게 추가 학습하는 과정입니다."},
    {"question": "LoRA가 뭐야?", "answer": "LoRA는 원본 가중치를 고정하고 저랭크 어댑터만 학습하는 파라미터 효율적 파인튜닝 기법입니다."},
    {"question": "QLoRA가 뭐야?", "answer": "QLoRA는 기반 모델을 4비트로 양자화한 상태에서 LoRA 어댑터를 학습해 메모리를 절약하는 방법입니다."},
    {"question": "LoRA와 QLoRA의 차이는?", "answer": "LoRA는 일반 정밀도 기반 모델에 어댑터를 붙이고, QLoRA는 4비트 기반 모델에 어댑터를 붙여 학습 메모리를 더 줄입니다."},
    {"question": "양자화가 뭐야?", "answer": "양자화는 모델 가중치의 정밀도를 낮춰 파일 크기와 메모리 사용량을 줄이는 기술입니다."},
    {"question": "PTQ가 뭐야?", "answer": "PTQ는 학습이 끝난 모델에 추가 학습 없이 적용하는 사후 양자화입니다."},
    {"question": "4비트 양자화의 장점은?", "answer": "4비트 양자화는 모델 크기와 메모리 사용량을 크게 줄여 로컬 추론을 쉽게 만듭니다."},
    {"question": "8비트 양자화의 장점은?", "answer": "8비트 양자화는 FP16보다 메모리를 줄이면서 일반적으로 4비트보다 품질 저하가 작습니다."},
    {"question": "GGUF가 뭐야?", "answer": "GGUF는 llama.cpp가 모델과 메타데이터를 효율적으로 저장하고 불러오는 파일 형식입니다."},
    {"question": "llama.cpp가 뭐야?", "answer": "llama.cpp는 GGUF 모델을 CPU와 다양한 GPU에서 효율적으로 추론하는 오픈소스 프로젝트입니다."},
    {"question": "Q4_K_M이 뭐야?", "answer": "Q4_K_M은 llama.cpp에서 크기와 품질의 균형이 좋은 혼합 4비트 양자화 방식입니다."},
    {"question": "어댑터가 뭐야?", "answer": "어댑터는 기반 모델에 추가되어 특정 작업을 학습하는 작은 수의 파라미터입니다."},
    {"question": "모델 병합은 왜 해?", "answer": "모델 병합은 LoRA 어댑터 가중치를 기반 모델에 합쳐 독립적인 하나의 모델로 만들기 위해 수행합니다."},
    {"question": "학습률이 뭐야?", "answer": "학습률은 한 번의 업데이트에서 모델 파라미터를 얼마나 크게 변경할지 정하는 값입니다."},
    {"question": "에폭이 뭐야?", "answer": "에폭은 전체 학습 데이터셋을 한 번 모두 학습한 횟수입니다."},
    {"question": "배치 크기가 뭐야?", "answer": "배치 크기는 한 번의 학습 단계에서 동시에 처리하는 샘플 수입니다."},
    {"question": "손실 함수가 뭐야?", "answer": "손실 함수는 모델 예측과 정답의 차이를 숫자로 측정하는 함수입니다."},
    {"question": "과적합이 뭐야?", "answer": "과적합은 모델이 학습 데이터에 지나치게 맞춰져 새로운 데이터에서 성능이 낮아지는 현상입니다."},
    {"question": "학습 데이터와 테스트 데이터는 왜 나눠?", "answer": "학습에 사용하지 않은 데이터로 일반화 성능을 공정하게 측정하기 위해 나눕니다."},
    {"question": "토크나이저가 뭐야?", "answer": "토크나이저는 텍스트를 모델이 처리할 수 있는 토큰 ID로 변환합니다."},
    {"question": "추론이 뭐야?", "answer": "추론은 학습이 끝난 모델에 입력을 넣어 예측이나 답변을 생성하는 과정입니다."},
    {"question": "GPU 메모리를 왜 측정해?", "answer": "모델이 실제 하드웨어에서 실행 가능한지와 양자화의 자원 절감 효과를 확인하기 위해 측정합니다."},
    {"question": "토큰 생성 속도는 어떻게 표시해?", "answer": "일반적으로 초당 생성한 토큰 수인 tokens per second로 표시합니다."},
    {"question": "양자화 후 품질도 확인해야 해?", "answer": "네. 크기와 속도뿐 아니라 같은 평가 질문에서 정확도가 유지되는지도 함께 확인해야 합니다."},
    {"question": "재현 가능한 실험이란?", "answer": "같은 설정과 데이터와 시드를 사용했을 때 다른 사람도 같은 절차를 다시 실행할 수 있는 실험입니다."},
]

TEST_CASES = [
    {"question": "LoRA를 한 문장으로 설명해줘.", "keywords": ["저랭크", "어댑터"]},
    {"question": "QLoRA가 메모리를 아끼는 이유는?", "keywords": ["4비트", "LoRA"]},
    {"question": "학습이 끝난 다음 하는 양자화는 뭐라고 해?", "keywords": ["PTQ", "사후"]},
    {"question": "GGUF 파일은 어디에서 사용해?", "keywords": ["llama.cpp"]},
    {"question": "Q4_K_M을 선택하는 이유는?", "keywords": ["크기", "품질"]},
    {"question": "모델을 양자화하면 무엇을 같이 비교해야 해?", "keywords": ["메모리", "품질"]},
]

print("학습 샘플:", len(TRAIN_PAIRS))
print("테스트 샘플:", len(TEST_CASES))


### 4. Qwen 채팅 형식으로 데이터 전처리

프롬프트 토큰의 label은 `-100`으로 가려서 답변 부분만 loss에 반영합니다.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def encode_example(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = prompt_text + example["answer"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]
    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    labels = list(encoded["input_ids"])
    for index in range(min(len(prompt_ids), len(labels))):
        labels[index] = -100
    encoded["labels"] = labels
    return encoded

train_dataset = Dataset.from_list(TRAIN_PAIRS).map(
    encode_example,
    remove_columns=["question", "answer"],
)

assert len(train_dataset) == len(TRAIN_PAIRS)
assert any(label != -100 for label in train_dataset[0]["labels"])
print(train_dataset)
print("첫 샘플 토큰 수:", len(train_dataset[0]["input_ids"]))


### 5. 공통 학습 함수

두 실험은 모델 로딩 방식만 다르고 LoRA rank, epoch, learning rate, batch 조건은 동일하게 유지합니다.


In [ ]:
import gc
import json
import time

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

training_results = []

def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def train_adapter(run_name, use_qlora, adapter_dir):
    cleanup_cuda()
    torch.cuda.reset_peak_memory_stats()

    if use_qlora:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=quantization_config,
            device_map={"": 0},
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=COMPUTE_DTYPE,
            low_cpu_mem_usage=True,
        ).to("cuda")

    model.config.use_cache = False

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_RANK * 2,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    trainable, total = model.get_nb_trainable_parameters()
    model.print_trainable_parameters()

    arguments = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"{run_name}_checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        fp16=not BF16_SUPPORTED,
        bf16=BF16_SUPPORTED,
        optim="adamw_torch",
        seed=SEED,
        data_seed=SEED,
    )
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )
    trainer = Trainer(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        data_collator=collator,
    )

    started = time.perf_counter()
    train_output = trainer.train()
    torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - started

    peak_allocated_gb = torch.cuda.max_memory_allocated() / (1024**3)
    peak_reserved_gb = torch.cuda.max_memory_reserved() / (1024**3)

    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    result = {
        "method": run_name,
        "base_precision": "4-bit NF4" if use_qlora else str(COMPUTE_DTYPE).replace("torch.", ""),
        "trainable_parameters": int(trainable),
        "total_parameters": int(total),
        "trainable_percent": 100 * trainable / total,
        "train_loss": float(train_output.metrics["train_loss"]),
        "training_time_sec": elapsed_sec,
        "peak_allocated_gpu_gb": peak_allocated_gb,
        "peak_reserved_gpu_gb": peak_reserved_gb,
        "adapter_size_mb": sum(
            path.stat().st_size for path in adapter_dir.rglob("*") if path.is_file()
        ) / (1024**2),
    }
    training_results.append(result)

    del trainer, model, train_output
    cleanup_cuda()
    return result


### 6. LoRA Fine-Tuning


In [ ]:
lora_result = train_adapter(
    run_name="LoRA",
    use_qlora=False,
    adapter_dir=ADAPTER_LORA_DIR,
)
lora_result


### 7. QLoRA Fine-Tuning


In [ ]:
qlora_result = train_adapter(
    run_name="QLoRA",
    use_qlora=True,
    adapter_dir=ADAPTER_QLORA_DIR,
)
qlora_result


### 8. LoRA와 QLoRA 학습 결과 비교


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

training_df = pd.DataFrame(training_results)
training_df.to_csv(RESULTS_DIR / "training_comparison.csv", index=False)
display(training_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
training_df.plot(
    x="method",
    y="training_time_sec",
    kind="bar",
    legend=False,
    title="Training time (seconds)",
    ax=axes[0],
    color=["#4C78A8", "#F58518"],
)
training_df.plot(
    x="method",
    y="peak_reserved_gpu_gb",
    kind="bar",
    legend=False,
    title="Peak reserved GPU memory (GB)",
    ax=axes[1],
    color=["#4C78A8", "#F58518"],
)
for axis in axes:
    axis.set_xlabel("")
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


#### 학습 결과 해석 방법

위 표에서는 먼저 두 실험의 trainable parameter 수가 유사한지 확인한다. 두 방법 모두 LoRA adapter를 학습하므로 학습 가능 파라미터 비율은 전체 모델에 비해 매우 작아야 한다. 차이가 크게 나타난다면 target module이나 rank 설정이 서로 다른지 확인해야 한다.

다음으로 peak reserved GPU memory를 비교한다. QLoRA는 기반 모델을 4비트 NF4로 불러오기 때문에 일반 LoRA보다 낮은 메모리 사용량이 예상된다. 다만 모델 크기가 작거나 Colab의 CUDA allocator가 캐시를 유지하면 이론적인 4분의 1만큼 정확히 감소하지 않을 수 있다.

학습 시간은 반드시 QLoRA가 더 빠르다고 단정할 수 없다. 4비트 양자화와 역양자화 계산 비용, GPU 종류, 커널 지원 여부에 따라 QLoRA가 비슷하거나 더 느릴 수 있다. 따라서 실제 측정값을 그대로 보고하고 메모리 절감과 속도 변화를 분리해 해석한다.

마지막으로 train loss는 LoRA와 QLoRA의 우열을 단정하는 절대 지표가 아니다. 동일한 데이터와 step에서 학습이 발산하지 않았는지 확인하는 용도로 사용하고, 실제 품질은 뒤의 고정 평가 질문으로 추가 확인한다.


### 9. QLoRA 어댑터를 기반 모델에 병합

GGUF 변환 전에 QLoRA 어댑터를 FP16/BF16 기반 Qwen 모델에 병합합니다.


In [ ]:
from peft import PeftModel

cleanup_cuda()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_QLORA_DIR)
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
)
tokenizer.save_pretrained(MERGED_DIR)

print("병합 모델 저장:", MERGED_DIR)
print("병합 모델 파일:")
for path in sorted(MERGED_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name}: {path.stat().st_size / (1024**2):.1f} MB")

del base_model, peft_model, merged_model
cleanup_cuda()


### PTQ 및 GGUF 변환 과정 설명

QLoRA 학습이 끝났을 때 저장되는 파일은 전체 모델이 아니라 adapter다. llama.cpp에서 단독 모델로 실행하려면 다음 순서가 필요하다.

Qwen base + QLoRA adapter → merged HF model → F16 GGUF → Q4_K_M GGUF

F16 GGUF는 양자화 전 기준점이다. Q4_K_M은 텐서 특성에 따라 서로 다른 K-quant 유형을 혼합해 크기와 품질 사이의 균형을 노리는 4비트 계열 방식이다. 양자화는 F16 GGUF에서 한 번만 수행하며, 이미 양자화된 파일을 다시 양자화하지 않는다. 재양자화는 누적 오차로 품질을 더 떨어뜨릴 수 있기 때문이다.

두 GGUF는 같은 병합 모델에서 파생되므로 파일 정밀도 외의 조건이 같다. 이후 비교에서도 같은 prompt, context, 생성 길이, temperature와 GPU offload 설정을 사용한다.


### 10. llama.cpp 빌드, GGUF 변환, Q4_K_M PTQ

이 단계에서 먼저 고정밀 `qwen-f16.gguf`를 만든 뒤, 학습 없이 `Q4_K_M`으로 양자화합니다. 이것이 과제의 Post-Training Quantization 단계입니다.


In [ ]:
import shlex

LLAMA_CPP_DIR = Path("/content/llama.cpp")
LLAMA_BUILD_DIR = LLAMA_CPP_DIR / "build"
LLAMA_CLI = LLAMA_BUILD_DIR / "bin" / "llama-cli"
LLAMA_QUANTIZE = LLAMA_BUILD_DIR / "bin" / "llama-quantize"
LLAMA_BENCH = LLAMA_BUILD_DIR / "bin" / "llama-bench"

F16_GGUF = GGUF_DIR / "qwen-f16.gguf"
Q4_GGUF = GGUF_DIR / "qwen-q4_k_m.gguf"

def run_command(command, cwd=None, tail_chars=4000):
    print("$", " ".join(shlex.quote(str(part)) for part in command))
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if completed.stdout:
        print(completed.stdout[-tail_chars:])
    if completed.returncode != 0:
        raise RuntimeError(f"명령 실패(returncode={completed.returncode})")
    return completed

if not LLAMA_CPP_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp.git",
        LLAMA_CPP_DIR,
    ])

requirements_file = (
    LLAMA_CPP_DIR
    / "requirements"
    / "requirements-convert_hf_to_gguf.txt"
)
run_command([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", requirements_file,
])

run_command([
    "cmake",
    "-S", LLAMA_CPP_DIR,
    "-B", LLAMA_BUILD_DIR,
    "-DGGML_CUDA=ON",
    "-DCMAKE_BUILD_TYPE=Release",
])
run_command([
    "cmake",
    "--build", LLAMA_BUILD_DIR,
    "--config", "Release",
    "--target", "llama-quantize", "llama-cli", "llama-bench",
    "-j", "2",
])

for output_file in [F16_GGUF, Q4_GGUF]:
    if output_file.exists():
        output_file.unlink()

run_command([
    sys.executable,
    LLAMA_CPP_DIR / "convert_hf_to_gguf.py",
    MERGED_DIR,
    "--outfile", F16_GGUF,
    "--outtype", "f16",
])
run_command([
    LLAMA_QUANTIZE,
    F16_GGUF,
    Q4_GGUF,
    "Q4_K_M",
])

llama_commit = subprocess.check_output(
    ["git", "-C", LLAMA_CPP_DIR, "rev-parse", "HEAD"],
    text=True,
).strip()
print("llama.cpp commit:", llama_commit)


## Checks

### 11. GGUF 파일 크기 확인


## 양자화 전후 평가 방법

### 1. 파일 크기

저장 공간 감소율은 (F16 크기 - Q4 크기) / F16 크기 × 100으로 계산한다.

### 2. 추론 속도

llama-bench를 사용해 prompt processing과 text generation을 분리해 측정한다. 보고서에서는 답변 생성과 직접 연결되는 text generation의 tokens/s를 주요 속도 지표로 사용한다. 측정 편차를 줄이기 위해 각 조건을 여러 번 반복한다.

### 3. 추론 메모리

Colab 커널이 이미 점유한 GPU 메모리를 기준값으로 측정한 뒤 llama.cpp 프로세스 실행 중 증가한 최대 메모리를 기록한다. 이는 절대적인 모델 메모리와 완전히 같지는 않지만, 동일 세션에서 F16과 Q4_K_M을 비교하는 상대 지표로 사용할 수 있다.

### 4. 답변 품질

양자화 모델은 크기가 작아지는 대신 일부 정보 손실이 생길 수 있다. 고정된 한국어 질문에 temperature 0으로 답하게 하고 기대 핵심어 포함률을 계산한다. 자동 점수만으로 판단하지 않고 실제 생성 문장을 함께 확인한다.


In [ ]:
model_files = {
    "FP16": F16_GGUF,
    "Q4_K_M": Q4_GGUF,
}

size_rows = []
for precision, model_path in model_files.items():
    if not model_path.exists():
        raise FileNotFoundError(model_path)
    size_rows.append({
        "precision": precision,
        "file": model_path.name,
        "file_size_mb": model_path.stat().st_size / (1024**2),
    })

size_df = pd.DataFrame(size_rows)
fp16_size = size_df.loc[size_df["precision"] == "FP16", "file_size_mb"].iloc[0]
q4_size = size_df.loc[size_df["precision"] == "Q4_K_M", "file_size_mb"].iloc[0]
size_reduction_pct = (fp16_size - q4_size) / fp16_size * 100

display(size_df.round(2))
print(f"파일 크기 감소율: {size_reduction_pct:.2f}%")


### 12. llama-bench로 동일 조건 속도 비교

각 모델에서 prompt processing 128토큰과 text generation 128토큰을 3회 반복합니다. 결과의 `avg_ts`는 초당 토큰 수입니다.


In [ ]:
benchmark_command = [
    LLAMA_BENCH,
    "-m", F16_GGUF,
    "-m", Q4_GGUF,
    "-p", "128",
    "-n", "128",
    "-r", "3",
    "-ngl", "99",
    "-o", "json",
]
benchmark_process = subprocess.run(
    [str(part) for part in benchmark_command],
    text=True,
    capture_output=True,
)
if benchmark_process.returncode != 0:
    print(benchmark_process.stderr[-4000:])
    raise RuntimeError("llama-bench 실행 실패")

benchmark_raw = json.loads(benchmark_process.stdout)
(RESULTS_DIR / "llama_bench_raw.json").write_text(
    json.dumps(benchmark_raw, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

benchmark_rows = []
for row in benchmark_raw:
    filename = Path(row["model_filename"]).name
    precision = "Q4_K_M" if "q4" in filename.lower() else "FP16"
    test_name = (
        f"pp{row['n_prompt']}"
        if row["n_prompt"] > 0
        else f"tg{row['n_gen']}"
    )
    benchmark_rows.append({
        "precision": precision,
        "test": test_name,
        "tokens_per_sec": row["avg_ts"],
        "stddev_tokens_per_sec": row["stddev_ts"],
        "backend": row["backends"],
        "gpu_info": row["gpu_info"],
    })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(RESULTS_DIR / "llama_bench_summary.csv", index=False)
display(benchmark_df.round(2))


### 13. 단일 추론과 추가 GPU 메모리 측정

노트북 커널이 이미 사용하는 GPU 메모리를 기준값으로 잡고, llama.cpp 프로세스 실행 중 추가로 사용된 최대 GPU 메모리를 측정합니다.


In [ ]:
import time

def current_compute_gpu_memory_mb():
    completed = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=used_gpu_memory",
            "--format=csv,noheader,nounits",
        ],
        text=True,
        capture_output=True,
    )
    values = []
    for line in completed.stdout.splitlines():
        value = line.strip().split()[0] if line.strip() else ""
        try:
            values.append(float(value))
        except ValueError:
            pass
    return sum(values)

def run_single_turn(model_path, prompt, max_tokens=96, monitor_memory=True):
    command = [
        str(LLAMA_CLI),
        "-m", str(model_path),
        "-p", prompt,
        "-sys", SYSTEM_PROMPT,
        "-st",
        "-n", str(max_tokens),
        "-ngl", "99",
        "--temp", "0",
        "--no-display-prompt",
        "--no-show-timings",
        "-co", "off",
    ]

    baseline_mb = current_compute_gpu_memory_mb()
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    peak_total_mb = baseline_mb
    while process.poll() is None:
        if monitor_memory:
            peak_total_mb = max(
                peak_total_mb,
                current_compute_gpu_memory_mb(),
            )
        time.sleep(0.05)
    output, _ = process.communicate()
    elapsed_sec = time.perf_counter() - started

    if process.returncode != 0:
        raise RuntimeError(output[-4000:])

    return {
        "answer": output.strip(),
        "elapsed_sec": elapsed_sec,
        "peak_added_gpu_mb": max(0.0, peak_total_mb - baseline_mb),
    }

memory_prompt = "LoRA와 QLoRA의 차이를 간단히 설명해줘."
inference_rows = []

for precision, model_path in model_files.items():
    result = run_single_turn(
        model_path,
        memory_prompt,
        max_tokens=96,
        monitor_memory=True,
    )
    inference_rows.append({
        "precision": precision,
        "elapsed_sec": result["elapsed_sec"],
        "peak_added_gpu_mb": result["peak_added_gpu_mb"],
        "answer": result["answer"][-1000:],
    })
    print(f"\n[{precision}]")
    print(result["answer"][-1000:])

inference_df = pd.DataFrame(inference_rows)
inference_df.to_csv(RESULTS_DIR / "single_turn_inference.csv", index=False)
display(inference_df[["precision", "elapsed_sec", "peak_added_gpu_mb"]].round(2))


### 14. 고정 질문으로 양자화 전후 답변 품질 비교

각 질문의 핵심어 포함 비율을 간단한 자동 점수로 사용합니다. 최종 보고서에는 자동 점수와 함께 실제 답변 예시도 넣으세요.


In [ ]:
quality_rows = []

for precision, model_path in model_files.items():
    for case in TEST_CASES:
        result = run_single_turn(
            model_path,
            case["question"],
            max_tokens=72,
            monitor_memory=False,
        )
        answer = result["answer"]
        lowered_answer = answer.lower()
        hits = [
            keyword
            for keyword in case["keywords"]
            if keyword.lower() in lowered_answer
        ]
        quality_rows.append({
            "precision": precision,
            "question": case["question"],
            "expected_keywords": ", ".join(case["keywords"]),
            "matched_keywords": ", ".join(hits),
            "keyword_score": len(hits) / len(case["keywords"]),
            "answer": answer[-1000:],
        })

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(RESULTS_DIR / "quality_comparison.csv", index=False)

quality_summary_df = (
    quality_df.groupby("precision", as_index=False)["keyword_score"]
    .mean()
    .rename(columns={"keyword_score": "keyword_accuracy"})
)
quality_summary_df["keyword_accuracy_pct"] = (
    quality_summary_df["keyword_accuracy"] * 100
)

display(quality_summary_df.round(2))
display(
    quality_df[
        ["precision", "question", "expected_keywords", "matched_keywords", "keyword_score"]
    ].round(2)
)


### 15. 최종 비교표와 그래프


In [ ]:
generation_speed = (
    benchmark_df[benchmark_df["test"].str.startswith("tg")]
    [["precision", "tokens_per_sec"]]
    .drop_duplicates("precision")
)
final_comparison_df = (
    size_df[["precision", "file_size_mb"]]
    .merge(generation_speed, on="precision", how="left")
    .merge(
        inference_df[["precision", "elapsed_sec", "peak_added_gpu_mb"]],
        on="precision",
        how="left",
    )
    .merge(
        quality_summary_df[["precision", "keyword_accuracy_pct"]],
        on="precision",
        how="left",
    )
)
final_comparison_df.to_csv(
    RESULTS_DIR / "final_quantization_comparison.csv",
    index=False,
)

display(final_comparison_df.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plots = [
    ("file_size_mb", "Model file size (MB)"),
    ("tokens_per_sec", "Generation speed (tokens/s)"),
    ("peak_added_gpu_mb", "Peak added GPU memory (MB)"),
]
for axis, (column, title) in zip(axes, plots):
    final_comparison_df.plot(
        x="precision",
        y=column,
        kind="bar",
        legend=False,
        ax=axis,
        title=title,
        color=["#4C78A8", "#F58518"],
    )
    axis.set_xlabel("")
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "quantization_comparison.png", dpi=160)
plt.show()

fp16_row = final_comparison_df.set_index("precision").loc["FP16"]
q4_row = final_comparison_df.set_index("precision").loc["Q4_K_M"]

memory_reduction_pct = (
    (fp16_row["peak_added_gpu_mb"] - q4_row["peak_added_gpu_mb"])
    / fp16_row["peak_added_gpu_mb"]
    * 100
    if fp16_row["peak_added_gpu_mb"] > 0
    else float("nan")
)
speed_change_pct = (
    (q4_row["tokens_per_sec"] - fp16_row["tokens_per_sec"])
    / fp16_row["tokens_per_sec"]
    * 100
)

print(f"파일 크기 감소율: {size_reduction_pct:.2f}%")
print(f"추가 GPU 메모리 감소율: {memory_reduction_pct:.2f}%")
print(f"생성 속도 변화율: {speed_change_pct:.2f}%")
print(
    "품질 점수 변화:",
    f"{fp16_row['keyword_accuracy_pct']:.2f}% → "
    f"{q4_row['keyword_accuracy_pct']:.2f}%",
)


### 16. 측정값 기반 상세 결과 해석

다음 셀은 실제 실행 결과를 이용해 보고서에 활용할 수 있는 상세 해석문을 자동으로 만든다. 수치가 실행 결과와 연결되어 있으므로 임의의 예상값을 작성하지 않는다.


In [ ]:
training_lookup = training_df.set_index("method")
lora_train = training_lookup.loc["LoRA"]
qlora_train = training_lookup.loc["QLoRA"]

training_memory_reduction_pct = (
    (lora_train["peak_reserved_gpu_gb"] - qlora_train["peak_reserved_gpu_gb"])
    / lora_train["peak_reserved_gpu_gb"]
    * 100
)
quality_delta_pp = (
    q4_row["keyword_accuracy_pct"] - fp16_row["keyword_accuracy_pct"]
)

detailed_interpretation = f"""
# 상세 실험 결과 해석

## 1. LoRA와 QLoRA Fine-Tuning

동일한 Qwen 기반 모델과 동일한 rank={LORA_RANK}, learning rate={LEARNING_RATE},
epoch={EPOCHS} 조건에서 LoRA와 QLoRA를 학습했다. LoRA의 최종 train loss는
{lora_train['train_loss']:.4f}, QLoRA의 최종 train loss는
{qlora_train['train_loss']:.4f}로 측정됐다.

LoRA의 peak reserved GPU memory는 {lora_train['peak_reserved_gpu_gb']:.2f}GB,
QLoRA는 {qlora_train['peak_reserved_gpu_gb']:.2f}GB였다. 이번 실행에서는
QLoRA가 LoRA 대비 학습 GPU 메모리를 약 {training_memory_reduction_pct:.2f}% 줄였다.
이는 QLoRA가 기반 모델을 4비트 NF4로 보관하고 LoRA adapter만 학습했기 때문으로
해석할 수 있다.

학습 시간은 LoRA {lora_train['training_time_sec']:.2f}초,
QLoRA {qlora_train['training_time_sec']:.2f}초였다. 이 값은 Colab GPU 종류와
4비트 연산 커널의 영향을 받으므로, 메모리 절감과 동일한 비율로 속도가
향상된다고 일반화할 수는 없다.

## 2. Post-Training Quantization

QLoRA adapter를 기반 모델에 병합한 뒤 F16 GGUF와 Q4_K_M GGUF를 생성했다.
F16 파일은 {fp16_row['file_size_mb']:.2f}MB, Q4_K_M 파일은
{q4_row['file_size_mb']:.2f}MB로 측정됐다. Q4_K_M PTQ를 통해 파일 크기가
약 {size_reduction_pct:.2f}% 감소했다.

이 단계는 QLoRA 학습 중 사용한 4비트 로딩과 별개의 사후 양자화다.
학습 완료 adapter가 병합된 동일 모델에서 F16과 Q4_K_M을 만들었기 때문에,
두 결과의 차이는 주로 GGUF 가중치 정밀도 차이에서 발생한다.

## 3. llama.cpp 추론 성능과 메모리

동일한 llama.cpp 빌드와 동일한 생성 조건에서 F16의 생성 속도는
{fp16_row['tokens_per_sec']:.2f} tokens/s, Q4_K_M은
{q4_row['tokens_per_sec']:.2f} tokens/s였다. 생성 속도 변화율은
{speed_change_pct:.2f}%였다.

단일 추론 중 추가 peak GPU memory는 F16
{fp16_row['peak_added_gpu_mb']:.2f}MB, Q4_K_M
{q4_row['peak_added_gpu_mb']:.2f}MB로 측정됐다. 메모리 값은 Colab 커널과
GPU 드라이버의 영향을 받을 수 있으므로 절대값보다 동일 세션에서의 상대적
차이를 중심으로 해석한다.

## 4. 답변 품질

고정 질문의 핵심어 포함 정확도는 F16
{fp16_row['keyword_accuracy_pct']:.2f}%, Q4_K_M
{q4_row['keyword_accuracy_pct']:.2f}%였다. 양자화 후 변화는
{quality_delta_pp:.2f}%p였다. 이 평가는 소규모 질문 집합을 사용하므로 모델의
일반적인 언어 능력 전체를 대표하지 않는다. 다만 이번 과제 범위에서는
양자화 전후 핵심 개념 보존 여부와 실제 답변 차이를 확인하는 근거로 사용할 수 있다.

## 5. 종합 결론

이번 실험은 LoRA와 QLoRA Fine-Tuning, adapter 병합, GGUF 변환,
Q4_K_M Post-Training Quantization, llama.cpp 추론까지 하나의 흐름으로
재현했다. QLoRA는 학습 단계의 GPU 메모리를 절약했고, Q4_K_M은 배포 단계의
파일 크기와 추론 메모리를 줄였다. 최종 모델 선택에서는 크기와 속도만이 아니라
답변 품질 변화도 함께 고려해야 한다.
""".strip()

print(detailed_interpretation)
(RESULTS_DIR / "detailed_interpretation.md").write_text(
    detailed_interpretation,
    encoding="utf-8",
)


### 실험의 한계

1. 학습 데이터가 소규모이므로 모델의 일반 능력 향상보다 파이프라인 동작 확인에 초점이 있다.
2. 테스트 질문 수도 제한적이어서 양자화 품질을 통계적으로 단정하기 어렵다.
3. GPU 메모리와 속도는 Colab에서 할당된 GPU 종류 및 당시 시스템 상태에 영향을 받는다.
4. LoRA와 QLoRA의 loss가 비슷하더라도 실제 생성 품질이 반드시 같다는 의미는 아니다.
5. 더 정확한 품질 비교를 위해서는 별도 평가셋의 perplexity, task accuracy 또는 사람 평가가 필요하다.

그럼에도 같은 모델·데이터·하이퍼파라미터와 동일한 llama.cpp 런타임을 사용했기 때문에, 이번 실습 범위에서 LoRA/QLoRA 및 F16/Q4_K_M의 상대적 차이를 확인하는 데에는 의미가 있다.


### 17. 결과 파일과 어댑터 압축

GGUF 파일은 크므로 그대로 `/content/week9_outputs/gguf`에 남깁니다. 제출용 ZIP에는 비교 결과, 그래프, LoRA/QLoRA 어댑터를 포함합니다.


In [ ]:
import shutil

submission_dir = Path("/content/week9_submission")
if submission_dir.exists():
    shutil.rmtree(submission_dir)
submission_dir.mkdir(parents=True)

shutil.copytree(RESULTS_DIR, submission_dir / "results")
shutil.copytree(ADAPTER_LORA_DIR, submission_dir / "lora_adapter")
shutil.copytree(ADAPTER_QLORA_DIR, submission_dir / "qlora_adapter")

environment_info = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "compute_dtype": str(COMPUTE_DTYPE),
    "gpu": gpu_check.stdout.strip(),
    "llama_cpp_commit": llama_commit,
    "f16_gguf": str(F16_GGUF),
    "q4_gguf": str(Q4_GGUF),
}
(submission_dir / "environment.json").write_text(
    json.dumps(environment_info, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 9주차 Qwen Fine-Tuning 및 양자화 결과",
    "",
    f"- 모델: {MODEL_ID}",
    f"- LoRA rank: {LORA_RANK}",
    f"- Epochs: {EPOCHS}",
    f"- FP16 GGUF 크기: {fp16_row['file_size_mb']:.2f} MB",
    f"- Q4_K_M GGUF 크기: {q4_row['file_size_mb']:.2f} MB",
    f"- 파일 크기 감소율: {size_reduction_pct:.2f}%",
    f"- FP16 생성 속도: {fp16_row['tokens_per_sec']:.2f} tokens/s",
    f"- Q4_K_M 생성 속도: {q4_row['tokens_per_sec']:.2f} tokens/s",
    f"- 생성 속도 변화율: {speed_change_pct:.2f}%",
    f"- FP16 키워드 정확도: {fp16_row['keyword_accuracy_pct']:.2f}%",
    f"- Q4_K_M 키워드 정확도: {q4_row['keyword_accuracy_pct']:.2f}%",
    "",
    "## 결론",
    "",
    "Q4_K_M 사후 양자화는 모델 파일 크기와 메모리 사용량을 줄였다. "
    "동일한 llama.cpp 환경에서 속도와 답변 품질을 비교했으며, "
    "측정 결과를 바탕으로 자원 효율과 품질 사이의 절충을 확인했다.",
]
(submission_dir / "report_summary.md").write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

zip_path = shutil.make_archive(
    "/content/week9_submission",
    "zip",
    submission_dir,
)
print("제출용 압축 파일:", zip_path)
print("FP16 GGUF:", F16_GGUF)
print("Q4 GGUF:", Q4_GGUF)


## 회고

LoRA와 양자화를 직접 돌려보며, 학습과 배포에서 메모리 차이가 크다는 걸 느꼈다.


## Next Steps

제출 전에 다음 항목을 확인하세요.

- [ ] LoRA와 QLoRA 학습 셀의 실행 로그가 남아 있는가?
- [ ] LoRA/QLoRA 학습 시간, loss, GPU 메모리 표가 보이는가?
- [ ] `qwen-f16.gguf`와 `qwen-q4_k_m.gguf`가 생성됐는가?
- [ ] llama.cpp FP16/Q4 추론 답변이 모두 보이는가?
- [ ] 파일 크기, tokens/s, 추가 GPU 메모리, 품질 점수 비교표가 있는가?
- [ ] 결과를 임의로 쓰지 않고 실제 실행 값으로 보고서를 작성했는가?

최종 제출물 권장 구성:

1. 이 노트북의 실행 완료본
2. `week9_submission.zip`
3. 보고서 PDF 또는 `report_summary.md`
4. 필요할 경우 Q4 GGUF 파일 또는 저장 위치 링크

> 메모리 측정값은 Colab 런타임의 다른 프로세스 영향을 받을 수 있습니다. FP16과 Q4를 같은 런타임과 같은 옵션에서 연속 측정했다는 점을 보고서에 명시하세요.
